# Analyze trends in Winter & Summer Days, plus Tmin and Tmax

# **** BELOW NEEDS UPDATING

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cartopy import crs as ccrs, feature as cfeature
import netCDF4
from netCDF4 import Dataset
from datetime import datetime as dt

import scipy.stats as stats
from sklearn import linear_model
import statsmodels.api as sm
import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess as  sm_lowess
import pwlf #piecewise linear fits

# for confidence bands on lowess
import scipy.interpolate

# to get rid of runtime warnings when OLS fits hit a divide by zero - handled by setting R2 and p-vals appropriately
import warnings
warnings.filterwarnings('ignore')

## Load LSM for filtering

In [3]:
file_path = '../../../../Data/ERA5-global/ERA5-2023-09-01-CoordFixed-LSM.nc'
ds_lsm = xr.open_dataset(file_path)
ds_lsm

<xarray.Dataset> Size: 8MB
Dimensions:  (lat: 721, lon: 1440)
Coordinates:
  * lat      (lat) float32 3kB 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
  * lon      (lon) float32 6kB -180.0 -179.8 -179.5 -179.2 ... 179.2 179.5 179.8
Data variables:
    lsm      (lat, lon) float64 8MB ...

## * Build and write out the full data set from Fourier fits including the coastal margin boolean

In [10]:
%%time

# = '../../../../Data/ERA5-global/Analysis/New-Fourier/Seasons/'+str(input_year)+'_summer_winter_days.nc'

# takes ~4 min to load

# start with one year and lsm then coastal and add a coordinate for the year
year = 1961
input_path = '../../../../Data/ERA5-global/Analysis/New-Fourier/Winter/'+str(year)+'_winter_stats.nc'
ds_base = xr.open_dataset(input_path)
ds_base = ds_base.merge(ds_lsm) # add lsm
ds_base = ds_base.merge(results_ds) # add coastal flag
ds_base = ds_base.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))

# loop thru remaining years and also add time coord
input_years = np.arange(1962,2026,1)

for year in input_years:
    input_path = '../../../../Data/ERA5-global/Analysis/New-Fourier/Winter/'+str(year)+'_winter_stats.nc'
    ds_i = xr.open_dataset(input_path)
    ds_i = ds_i.merge(ds_lsm)
    ds_i = ds_i.merge(results_ds)
    ds_i = ds_i.expand_dims("time").assign_coords(time=("time", [dt(year,1,1)]))
    ds_base = ds_base.merge(ds_i)

ds_base

CPU times: user 1min 33s, sys: 2min 49s, total: 4min 23s
Wall time: 4min 59s


<xarray.Dataset> Size: 5GB
Dimensions:         (lat: 721, lon: 1440, time: 65)
Coordinates:
  * lat             (lat) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * lon             (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
  * time            (time) datetime64[us] 520B 1961-01-01 ... 2025-01-01
Data variables:
    WinterStart     (time, lat, lon) float64 540MB 88.0 88.0 88.0 ... 0.0 0.0
    WinterEnd       (time, lat, lon) float64 540MB 125.0 125.0 125.0 ... 0.0 0.0
    WinterTmax      (time, lat, lon) float64 540MB 94.0 94.0 94.0 ... 4.0 4.0
    WinterCold      (time, lat, lon) float64 540MB 70.38 70.38 70.38 ... 0.0 0.0
    WinterLength    (time, lat, lon) float64 540MB 38.0 38.0 38.0 ... 0.0 0.0
    WinterlikeDays  (time, lat, lon) float64 540MB 96.0 96.0 96.0 ... 9.0 9.0
    WinterRMSE      (time, lat, lon) float64 540MB 3.98 3.98 ... 3.641 3.641
    WinterMeanT     (time, lat, lon) float64 540MB 218.1 218.1 218.1 ... 0.0 0.0
    lsm             (time, lat, lon) float64 540MB 1.0 1.0 1.0 ... 0.0 0.0 0.0
    Coastal         (time, lat, lon) float64 540MB 0.0 0.0 0.0 ... 0.0 0.0 0.0

In [ ]:
    # add attributes
    first_wld.attrs["long_name"] = "First day with mean T2m at or below winter threshold for the year (DOY in [1,365])"
    first_wld.attrs["units"] = "Day of year"
    
    last_wld.attrs["long_name"] = "Last day with mean T2m at or below winter threshold for the year (DOY in [1,365])"
    first_wld.attrs["units"] = "Day of year"
    
    first_sld.attrs["long_name"] = "First day with mean T2m at or above summer threshold for the year (DOY in [1,365])"
    first_sld.attrs["units"] = "Day of year"
    
    last_sld.attrs["long_name"] = "Last day with mean T2m at or above summer threshold for the year (DOY in [1,365])"
    first_sld.attrs["units"] = "Day of year"
    
    day_of_max.attrs["long_name"] = "Day of max mean T2m during the year where summer is centered"
    day_of_max.attrs["units"] = "Day of year"
    
    t_max.attrs["long_name"] = "Max T2m during the year where summer is centered"
    t_max.attrs["units"] = "Degrees [K]"
    
    day_of_min.attrs["long_name"] = "Day of min mean T2m during the year where winter is centered"
    day_of_min.attrs["units"] = "Day of year"
    
    t_min.attrs["long_name"] = "Min T2m during the year where winter is centered"
    t_min.attrs["units"] = "Degrees [K]"

    winterlike_days.attrs["long_name"] = "Number of days during the year where winter is centered with mean T2m at or below threshold"
    winterlike_days.attrs["units"] = "Days"

    summerlike_days.attrs["long_name"] = "Number of days during the year where summer is centered with mean T2m at or above threshold"
    summerlike_days.attrs["units"] = "Days"
    

In [11]:
# re-add attributes which get lost on merge apparently
ds_base.WinterStart.attrs["long_name"] = "Start day of winter for the year (DOY in [1,365])"
ds_base.WinterStart.attrs["units"] = "Day of year"

ds_base.WinterEnd.attrs["long_name"] = "Last day of winter for the year (DOY in [1,365])"
ds_base.WinterEnd.attrs["units"] = "Day of year"

ds_base.WinterLength.attrs["long_name"] = "Duration of winter for the year"
ds_base.WinterLength.attrs["units"] = "Days"

ds_base.WinterlikeDays.attrs["long_name"] = "Number of days in the 12-month period with mean T2m at or below threshold"
ds_base.WinterlikeDays.attrs["long_name"] = "Days"

ds_base.WinterTmax.attrs["long_name"] = "Day of max mean temp during winter for the year"
ds_base.WinterTmax.attrs["units"] = "Day of year"

ds_base.WinterCold.attrs["long_name"] = "Accumulated cold during winter for the year"
ds_base.WinterCold.attrs["units"] = "Degree-Days"

ds_base.WinterRMSE.attrs["long_name"] = "Root mean squared error of Fourier fit"
ds_base.WinterRMSE.attrs["units"] = "Degrees [K]"

ds_base.WinterMeanT.attrs["long_name"] = "Mean temperature during the winter period"
ds_base.WinterMeanT.attrs["units"] = "Degrees [K]"

ds_base.lsm.attrs["long_name"] = "Land-sea mask - a measure of the amount of land in the cell"
ds_base.lsm.attrs["units"] = "(0 - 1)"

###
### also cast the Coastal as boolean?
###
ds_base.Coastal.values = ds_base.Coastal.values.astype(bool)
ds_base.Coastal.attrs["long_name"] = "Boolean value for whether a >30% land cell has at least one >80% water neighbor"
ds_base.Coastal.attrs["units"] = "true/false"

ds_base

<xarray.Dataset> Size: 5GB
Dimensions:         (lat: 721, lon: 1440, time: 65)
Coordinates:
  * lat             (lat) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * lon             (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
  * time            (time) datetime64[us] 520B 1961-01-01 ... 2025-01-01
Data variables:
    WinterStart     (time, lat, lon) float64 540MB 88.0 88.0 88.0 ... 0.0 0.0
    WinterEnd       (time, lat, lon) float64 540MB 125.0 125.0 125.0 ... 0.0 0.0
    WinterTmax      (time, lat, lon) float64 540MB 94.0 94.0 94.0 ... 4.0 4.0
    WinterCold      (time, lat, lon) float64 540MB 70.38 70.38 70.38 ... 0.0 0.0
    WinterLength    (time, lat, lon) float64 540MB 38.0 38.0 38.0 ... 0.0 0.0
    WinterlikeDays  (time, lat, lon) float64 540MB 96.0 96.0 96.0 ... 9.0 9.0
    WinterRMSE      (time, lat, lon) float64 540MB 3.98 3.98 ... 3.641 3.641
    WinterMeanT     (time, lat, lon) float64 540MB 218.1 218.1 218.1 ... 0.0 0.0
    lsm             (time, lat, lon) float64 540MB 1.0 1.0 1.0 ... 0.0 0.0 0.0
    Coastal         (time, lat, lon) bool 67MB False False False ... False False

In [12]:
# write out complete dataset with LSM and Coastal flags
output_path = '../../../../Data/ERA5-global/Analysis/New-Fourier/Winter/1961-2025_ALL_winter_stats.nc'
ds_base.to_netcdf(output_path)



In [13]:
! ls -al "../../../../Data/ERA5-global/Analysis/New-Fourier/Winter"

total 18062184
drwxr-xr-x@ 70 tedscott  staff        2240 Jul 15 14:33 .
drwx------@ 90 tedscott  staff        2880 Jul 19 13:07 ..
-rw-r--r--@  1 tedscott  staff        6148 Jun 15 12:00 .DS_Store
-rw-r--r--@  1 tedscott  staff    66478723 Jul 18 07:45 1961_winter_stats.nc
-rw-r--r--@  1 tedscott  staff  4926484300 Jul 20 10:35 1961-2025_ALL_winter_stats.nc
-rw-r--r--@  1 tedscott  staff    66478723 Jul 18 08:17 1962_winter_stats.nc
-rw-r--r--@  1 tedscott  staff    66478723 Jul 18 08:48 1963_winter_stats.nc
-rw-r--r--@  1 tedscott  staff    66478723 Jul 18 09:20 1964_winter_stats.nc
-rw-r--r--@  1 tedscott  staff    66478723 Jul 18 09:52 1965_winter_stats.nc
-rw-r--r--@  1 tedscott  staff    66478723 Jul 18 10:24 1966_winter_stats.nc
-rw-r--r--@  1 tedscott  staff    66478723 Jul 18 10:56 1967_winter_stats.nc
-rw-r--r--@  1 tedscott  staff    66478723 Jul 18 11:28 1968_winter_stats.nc
-rw-r--r--@  1 tedscott  staff    66478723 Jul 18 11:59 1969_winter_stats.nc
-rw-r--r--@  1 tedscott

# **********

# *** JUMP here after first running

In [14]:
ds = xr.open_dataset('../../../../Data/ERA5-global/Analysis/New-Fourier/Winter/1961-2025_ALL_winter_stats.nc')
ds

<xarray.Dataset> Size: 5GB
Dimensions:         (time: 65, lat: 721, lon: 1440)
Coordinates:
  * time            (time) datetime64[ns] 520B 1961-01-01 ... 2025-01-01
  * lat             (lat) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * lon             (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
Data variables:
    WinterStart     (time, lat, lon) float64 540MB ...
    WinterEnd       (time, lat, lon) float64 540MB ...
    WinterTmax      (time, lat, lon) float64 540MB ...
    WinterCold      (time, lat, lon) float64 540MB ...
    WinterLength    (time, lat, lon) float64 540MB ...
    WinterlikeDays  (time, lat, lon) float64 540MB ...
    WinterRMSE      (time, lat, lon) float64 540MB ...
    WinterMeanT     (time, lat, lon) float64 540MB ...
    lsm             (time, lat, lon) float64 540MB ...
    Coastal         (time, lat, lon) bool 67MB ...

## Day of Tmax vs first SLD, Day of Tmin vs first WLD (or subtract and see if the day is shifting relative to the first day